# GSEA - KEGG

In [ ]:
import os
import re
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import numpy as np
import polars as pl

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Resolve this analysis folder (paper/01_rna) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '00_build_metadata.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/01_rna')

DIR = _here()

GSEA_FILE   = DIR / 'results' / 'gsea_all_results.pq'   # <- 20_gsea_kegg.py
BRITE_CACHE = DIR / 'data' / 'kegg_brite_map.tsv'      # vendored; refetched if absent

# 4 sequential stage transitions
CONTRASTS = ['DE_vs_ESC', 'HB_vs_DE', 'iHEP_vs_HB', 'mHEP_vs_iHEP']
CLABELS   = ['ESC→DE', 'DE→HB', 'HB→iHEP', 'iHEP→mHEP']

# BRITE hierarchy
CAT_ORDER = ['Metabolism', 'Genetic Information Processing',
             'Environmental Information Processing', 'Cellular Processes',
             'Organismal Systems', 'Human Diseases', 'Drug Development']
KEEP_CATEGORIES = {'Environmental Information Processing', 'Cellular Processes'}
# Enrichr names that don't match a BRITE pathway name verbatim
ALIASES = {
    'LYSOSOME': ('Cellular Processes', 'Transport and catabolism'),
    'CORONAVIRUS DISEASE - COVID-19': ('Human Diseases', 'Infectious disease: viral'),
}

ROW_THRESH = 0.05   # include a pathway if FDR < this in >=1 transition

# Color: scalar = sign(NES) * -log10(FDR); RdBu_r over [-3, 3]
CMAP  = plt.get_cmap('PuOr_r')
CNORM = Normalize(-3.0, 3.0)

def signed_sig(nes, fdr):
    return np.sign(nes) * min(-np.log10(max(fdr, 1e-3)), 3.0)

## Load data — GSEA results + BRITE pathway hierarchy

In [ ]:
def _norm(s):  return re.sub(r"\s+", " ", s.strip().lower())
def _alnum(s): return re.sub(r"[^a-z0-9]", "", s.lower())

def load_brite_map():
    """pathway-name -> (category, subcategory) from KEGG BRITE br08901 (cached)."""
    if BRITE_CACHE.exists():
        return pl.read_csv(BRITE_CACHE, separator="\t")
    txt = urllib.request.urlopen("https://rest.kegg.jp/get/br:br08901",
                                 timeout=30).read().decode()
    cat = sub = None; rows = []
    for line in txt.splitlines():
        if not line or line[0] not in "ABC":
            continue
        body = re.sub(r"<[^>]+>", "", line[1:]).strip()
        if line[0] == "A":   cat = body
        elif line[0] == "B": sub = body
        elif line[0] == "C":
            m = re.match(r"(\d{5})\s+(.*)", body)
            if m:
                rows.append({"map": m.group(1), "pathway": m.group(2),
                             "category": cat, "subcategory": sub})
    brite = pl.DataFrame(rows)
    BRITE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    brite.write_csv(BRITE_CACHE, separator="\t")
    return brite

brite = load_brite_map()
by_norm  = {_norm(p): (c, s) for p, c, s in
            zip(brite["pathway"], brite["category"], brite["subcategory"])}
by_alnum = {_alnum(p): (c, s) for p, c, s in
            zip(brite["pathway"], brite["category"], brite["subcategory"])}

def annotate(term):
    if term in ALIASES:        return ALIASES[term]
    if _norm(term) in by_norm: return by_norm[_norm(term)]
    if _alnum(term) in by_alnum: return by_alnum[_alnum(term)]
    return ("(unmapped)", "(unmapped)")

print(f"BRITE pathways: {brite.height}")

In [ ]:
df = pl.read_parquet(GSEA_FILE).with_columns([
    pl.col("NES").cast(float),
    pl.col("FDR q-val").cast(float).alias("fdr"),
])
kegg_libs = [l for l in df["library"].unique().to_list() if l.startswith("KEGG_")]
if not kegg_libs:
    raise SystemExit("no KEGG_* library in the GSEA results — re-run 20_gsea_kegg.py")
KEGG_LIB = os.environ.get("KEGG_LIB") or max(
    kegg_libs, key=lambda l: int(re.search(r"KEGG_(\d{4})", l).group(1)))
print(f"KEGG library: {KEGG_LIB}  (available: {sorted(kegg_libs)})")

k = df.filter((pl.col("library") == KEGG_LIB)
              & pl.col("contrast").is_in(CONTRASTS))
terms = sorted(k["Term"].unique().to_list())
cidx = {c: j for j, c in enumerate(CONTRASTS)}
nes = np.full((len(terms), 4), np.nan)
fdr = np.full((len(terms), 4), np.nan)
for row in k.iter_rows(named=True):
    i = terms.index(row["Term"]); j = cidx[row["contrast"]]
    nes[i, j] = row["NES"]; fdr[i, j] = row["fdr"]

# include sig pathways in the kept categories; order by category then peak |NES|
recs = []
for i in range(len(terms)):
    if not (np.nansum(fdr[i] < ROW_THRESH) >= 1):
        continue
    cat, sub = annotate(terms[i])
    if cat not in KEEP_CATEGORIES:
        continue
    peak_j = int(np.nanargmax(np.abs(nes[i])))
    recs.append({"i": i, "term": terms[i], "category": cat,
                 "peak_j": peak_j, "peak_nes": float(nes[i, peak_j])})

crank = lambda c: CAT_ORDER.index(c) if c in CAT_ORDER else len(CAT_ORDER)
recs.sort(key=lambda r: (crank(r["category"]), r["peak_j"], -r["peak_nes"]))
n = len(recs)

cat_bounds = [0]
for j in range(1, n):
    if recs[j]["category"] != recs[j - 1]["category"]:
        cat_bounds.append(j)
cat_bounds.append(n)
cat_spans = [(cat_bounds[t], cat_bounds[t + 1], recs[cat_bounds[t]]["category"])
             for t in range(len(cat_bounds) - 1)]
print(f"{n} pathways (EIP + Cellular Processes, FDR<{ROW_THRESH} in ≥1 transition)")

## Plot

In [ ]:
mx = float(np.nanmax(np.abs(nes[[r["i"] for r in recs]])))
xmax = max(2.5, np.ceil(mx * 2) / 2)
xticks = [-2, 0, 2]

fig = plt.figure(figsize=(10.5, max(5, 0.20 * n + 1.9)))
gs = fig.add_gridspec(1, 6, width_ratios=[1.5, 1, 1, 1, 1, 0.13], wspace=0.10)
ax_cat = fig.add_subplot(gs[0, 0])
axes = []  # one panel per transition, sharing the pathway-row y-axis
for j in range(4):
    axj = fig.add_subplot(gs[0, 1 + j], sharey=(axes[0] if axes else None))
    axes.append(axj)
ax_cb = fig.add_subplot(gs[0, 5])

for j, axj in enumerate(axes):
    axj.axvline(0, color="0.55", lw=1, zorder=4)
    for r_i, r in enumerate(recs):
        i = r["i"]
        if np.isnan(nes[i, j]):
            continue
        axj.barh(r_i, width=nes[i, j], left=0, height=0.82,
                 color=CMAP(CNORM(signed_sig(nes[i, j], fdr[i, j]))),
                 edgecolor="none", linewidth=0.3, zorder=3)
    axj.set_xlim(-xmax, xmax)
    axj.set_ylim(-0.7, n - 0.3)
    axj.invert_yaxis()
    axj.set_title(CLABELS[j], fontsize=9)
    axj.set_xticks(xticks)
    axj.set_xticklabels([str(t) for t in xticks], fontsize=7)
    axj.set_xlabel("NES", fontsize=8)
    axj.tick_params(axis="y", length=0)
    for b in cat_bounds[1:-1]:
        axj.axhline(b - 0.5, color="black", lw=1.2)
    if j == 0:
        axj.set_yticks(range(n))
        axj.set_yticklabels([r["term"][:46] for r in recs], fontsize=6.5,
                            family="monospace")
    else:
        axj.tick_params(labelleft=False)

# category labels (left)
ax_cat.set_xlim(0, 1); ax_cat.set_ylim(-0.7, n - 0.3)
ax_cat.invert_yaxis(); ax_cat.axis("off")
cmap_cat = plt.get_cmap("tab10")
for ci, (a, b, name) in enumerate(cat_spans):
    mid = (a + b - 1) / 2
    wrapped = name.replace("Environmental Information Processing",
                           "Environmental\nInfo. Processing")
    ax_cat.text(0.95, mid, wrapped, ha="right", va="center", rotation=90,
                fontsize=8.5, fontweight="bold", color=cmap_cat(ci % 10))

# single diverging colorbar in true units: signed -log10(FDR), sign = NES direction
cb = fig.colorbar(ScalarMappable(norm=CNORM, cmap=CMAP), cax=ax_cb)
cb.set_ticks([-3, -2, -1, 0, 1, 2, 3])
cb.set_ticklabels(["−3", "−2", "−1", "0", "1", "2", "3"], fontsize=7)
cb.set_label("signed −log10(FDR q-val)", fontsize=8)
ax_cb.text(0.5, 1.015, "up", transform=ax_cb.transAxes, ha="center",
           va="bottom", fontsize=7, color="#b2182b")
ax_cb.text(0.5, -0.015, "down", transform=ax_cb.transAxes, ha="center",
           va="top", fontsize=7, color="#2166ac")

fig.suptitle(
    "KEGG GSEA across the 4 stage transitions — EIP + Cellular Processes\n"
    "bar = NES (per-panel axis),  color = signed −log10(FDR q-val), sign = NES direction "
    "(capped at ±3 = FDR≤1e-3)",
    fontsize=10, y=0.997)

In [ ]:
FIG_NAME = 'kegg_nes_bars_eip_cellular'
fig.savefig(DIR / 'figs' / f"{FIG_NAME}.png", dpi=200, bbox_inches="tight")
fig.savefig(DIR / 'figs' / f"{FIG_NAME}.pdf", bbox_inches="tight")